# Argo float profiles — quickstart

Pull a set of Argo float profiles (temperature, salinity, pressure) for a small
box in the North Atlantic and plot a temperature–depth profile. Argo is open
data — no credentials. This backend needs the `argopy` SDK:
`pip install earthlens[argo]`.

## Setup

`earthlens` provides the unified `EarthLens` entry point; `matplotlib` draws the
profile. Argo is a *tabular* backend — `download()` returns a long-format
`DataFrame` (one row per measured level) and also writes it to disk.

In [ ]:
import matplotlib.pyplot as plt

from earthlens.earthlens import EarthLens

## 1 — Fetch a region of profiles

Name the family parameters you're interested in (`TEMP`, `PSAL`); the bounding box,
time window and depth range build the Argo region query. We use `source="gdac"` (the
Global Data Assembly Centre) as the data backend. A region fetch returns the whole
family (here `PRES`/`TEMP`/`PSAL` plus QC/error columns) — the names validate intent,
they don't subset the columns.

### Build the request

Describe what we want — the core physical parameters, a two-week January 2020
window, and a small box south of Nova Scotia. Building the request first keeps it
easy to read and re-run.

In [ ]:
floats = EarthLens(
    data_source="argo",
    variables=["TEMP", "PSAL"],
    start="2020-01-01",
    end="2020-01-15",
    aoi=[-60.0, 40.0, -55.0, 45.0],
    path="_nb_out",
    source="gdac",
)

### Download into a DataFrame

`download()` realises the profiles into memory (and writes the backing CSV).
`head()` previews the first few measured levels.

In [ ]:
df = floats.download(progress_bar=False)
df.head()

## 2 — Plot a temperature–depth profile

Each float reports temperature at many pressure levels. Plot `TEMP` against `PRES`
(pressure in dbar ≈ depth in metres) with the y-axis inverted, so the surface is
at the top — the classic Argo profile view.

In [ ]:
ax = df.plot.scatter(x="TEMP", y="PRES", s=4, figsize=(4, 6))
ax.invert_yaxis()
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Pressure (dbar)")
ax.set_title("Argo temperature profiles — N. Atlantic, Jan 2020")
plt.tight_layout()

## 3 — A single float by WMO id

Instead of a region, a `float:` selector targets one float by its WMO id (the bbox
is then ignored). A `profile:<WMO>/<cycle>` selector narrows to one cycle.

In [ ]:
one_float = EarthLens(
    data_source="argo",
    variables=["float:6902746"],
    start="2020-01-01",
    end="2020-12-31",
    path="_nb_out",
    source="gdac",
).download(progress_bar=False)

one_float[["PLATFORM_NUMBER", "CYCLE_NUMBER", "TIME", "PRES", "TEMP", "PSAL"]].head()